# Environment 2: Frozen Lake

So sánh SARSA và Expected SARSA trên hai cấu hình deterministic/slippery với thiết kế kiểm soát theo **environment-step budget**. Evaluation được tính chính xác từ transition model; variance được phân rã thành action-sampling variance và transition/reward variance.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/Briannguyen3106/project1.git'
ROOT = Path('/kaggle/working/project1')

if ROOT.exists():
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY, str(ROOT)], check=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.Environment2 as experiment
experiment.OUTPUT_DIR = Path('/kaggle/working/results/figures/environment2_frozen_lake')
run_experiment = experiment.run_experiment
OUTPUT_DIR = experiment.OUTPUT_DIR
print(f'Project root: {ROOT}')
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
%cd /kaggle/working/project1
!pip install -r requirements.txt

## Chạy thí nghiệm chính

Mỗi tổ hợp setting/thuật toán chạy 30 seed và đúng 100.000 training transitions cho mỗi seed. Epsilon giảm theo cumulative environment steps, không theo episode.

In [ ]:
episodes_df, evaluations_df, q_tables = run_experiment(
    seeds=list(range(30)),
    step_budget=100_000,
)
print(f'Results saved to: {OUTPUT_DIR}')

## Sanity checks và kết quả cuối

Bảng đầu xác nhận mọi run dùng cùng interaction budget. Bảng thứ hai là exact greedy success probability, không chứa Monte Carlo evaluation noise.

In [ ]:
step_check = (
    episodes_df.groupby(['setting', 'algorithm', 'seed'])['cumulative_steps']
    .max()
    .groupby(['setting', 'algorithm'])
    .agg(['min', 'max'])
)
display(step_check)

final_evaluations = (
    evaluations_df
    .sort_values(['setting', 'algorithm', 'seed', 'cumulative_steps'])
    .groupby(['setting', 'algorithm', 'seed'], as_index=False)
    .tail(1)
)
display(
    final_evaluations.groupby(['setting', 'algorithm'])['eval_success_rate']
    .agg(['mean', 'std', 'median', 'min', 'max'])
)

## Variance decomposition

Các đại lượng dưới đây được tính có điều kiện trên state-action và Q-table hiện tại bằng transition model của Frozen Lake. Expected SARSA có action-sampling variance bằng 0 theo định nghĩa; SARSA có thêm thành phần này.

In [ ]:
late_training = episodes_df[episodes_df['cumulative_steps'] > 80_000]
variance_summary = (
    late_training.groupby(['setting', 'algorithm'])[[
        'action_sampling_variance',
        'transition_variance',
        'model_target_variance',
    ]]
    .mean()
)
display(variance_summary)

In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/environment2_frozen_lake', 'zip', OUTPUT_DIR)
print(f'Archive ready: {archive}')